# 🧠 v0.0.2 · Ferrando entity-recognition test — properly controlled

Replication of [Ferrando et al. ICLR 2025 ("Do I Know This Entity?", arxiv:2411.14257)](https://arxiv.org/abs/2411.14257) on `caiovicentino1/qwen36-27b-sae-papergrade`, this time with the methodology controls v0.0.1 missed.

**v0.0.1 hit a tokenization confound**: synthetic Slavic-style "unknown" names were ~2× longer than famous "known" names. SAE features detected the surface difference, AUROC=1.0 was fake.

**What v0.0.2 fixes**, per the Ferrando 2024 paper + their [GitHub repo](https://github.com/javiferran/sae_entities):

1. **Same-distribution dataset**: both classes come from real Wikidata. We clone their entity JSONs (player/movie/city/song — all real entities).
2. **Model-defined labels**: known vs unknown is decided by **Qwen3.6-27B's own attribute recall**. Ask the model questions about each entity, score correctness. Famous-but-model-doesn't-know = unknown. Same-distribution surface forms.
3. **Discard the middle**: entities the model gets *partially* right are dropped — clean binary.
4. **Pile noise filter**: drop any feature active on >2% of random Pile tokens. **This is the explicit step that kills surface-confound features** like the ones that gave us fake AUROC=1.0 in v0.0.1.
5. **Single-latent scoring**: AUROC from one feature's raw activation magnitude (no LogReg multi-feature fitting). Matches Ferrando 2024 protocol.

**Cost**: ~$15 GPU + ~2h wall on RTX 6000 Pro.

**Stage gate**: AUROC ≥ 0.65 = strong · 0.55-0.65 = weak · < 0.55 = honest negative.

Reference: Ferrando 2024 reported **AUROC 0.732 on Gemma-2-2B-IT at L13**. Our 27B is much larger and reasoning-tuned — could go either way with proper controls.

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub datasets scikit-learn matplotlib tqdm
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 1. Config + load model + 3 SAEs

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
LAYERS        = [11, 31, 55]
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

# Sampling per entity type (4 types × 250 = 1000 candidate entities → after labelling ~300-500 surviving)
N_PER_TYPE    = 250
N_ATTRIBUTES_TO_TEST = 3   # ask 3 attribute questions per entity
MAX_GEN_TOKENS = 24        # short generations for attribute answers

# Pile noise filter
N_PILE_TOKENS = 2000
PILE_FILTER_THRESHOLD = 0.02   # drop features active on >2% of Pile tokens

# Eval
TRAIN_FRAC = 0.7
TOP_FEATURES_TO_TEST = 100   # we'll AUROC the top-100 by separation, report best single-latent
SEED = 0

import os, math, json, time, random, re
import numpy as np
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map='cuda',
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z

saes = {}
for layer in LAYERS:
    path = hf_hub_download(HF_SAE_REPO, f'sae_L{layer}_latest.safetensors')
    saes[layer] = TopKSAE(load_file(path), K).to(device).eval()
    print(f'  ✓ SAE L{layer}')

layer_mods = {layer: model.model.language_model.layers[layer] for layer in LAYERS}
print(f'\nready · vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 2. Pull Ferrando's pre-processed entity files

Direct download from `javiferran/sae_entities`. Real Wikidata entities with verified attributes. Same source for all classes — no synthetic name generation.

In [ ]:
import requests

FERRANDO_BASE = 'https://raw.githubusercontent.com/javiferran/sae_entities/main/dataset/processed/entities'
ENTITY_TYPES = ['player', 'movie', 'city', 'song']

raw_entities = {}
for t in ENTITY_TYPES:
    url = f'{FERRANDO_BASE}/{t}.json'
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    raw_entities[t] = r.json()
    print(f'  {t}: {len(raw_entities[t]):,} entities')

# Sample N_PER_TYPE per category (deterministic)
rng = random.Random(SEED)
candidates = []
for t in ENTITY_TYPES:
    pool = raw_entities[t]
    sample = rng.sample(pool, min(N_PER_TYPE, len(pool)))
    for ent in sample:
        candidates.append({'type': t, **ent})

print(f'\ncandidates total: {len(candidates)}')
print(f'sample: {candidates[0]["entity"]} ({candidates[0]["type"]})  attributes: '
      f'{[a["attribute_type"] for a in candidates[0]["attributes"][:3]]}')

## 3. Attribute-recall labelling — Qwen3.6-27B decides known vs unknown

For each entity, ask 3 attribute questions in chat format (Qwen3.6 is instruction-tuned). Score correctness by case-insensitive substring match between the ground-truth attribute value and the model's short answer.

- **Known**: ≥2/3 correct AND model didn't refuse on any
- **Unknown**: 0/3 correct AND model refused on at least 1 (or said "I don't know")
- **Middle**: anything else → discarded

This matches Ferrando's `discard-the-middle` protocol from `filter_known_unknown_wikidata.py`.

In [ ]:
from tqdm.auto import tqdm

ATTRIBUTE_TEMPLATES = {
    # type: { attribute_name: question }
    'player': {
        'place_birth':   "What is the place of birth of the basketball player '{entity}'? Answer with just the city name.",
        'team':          "What was a team that the basketball player '{entity}' played for? Answer with just the team name.",
        'height_cm':     "What is the height in cm of the basketball player '{entity}'? Answer with just a number.",
        'date_of_birth': "In what year was the basketball player '{entity}' born? Answer with just a year.",
        'draft_year':    "In what year was the basketball player '{entity}' drafted to the NBA? Answer with just a year.",
    },
    'movie': {
        'director':      "Who is the director of the movie '{entity}'? Answer with just the director's name.",
        'release_year':  "In what year was the movie '{entity}' released? Answer with just a year.",
        'genre':         "What is the genre of the movie '{entity}'? Answer with just the genre.",
    },
    'city': {
        'country':       "What country is the city of '{entity}' in? Answer with just the country name.",
        'continent':     "What continent is the city of '{entity}' on? Answer with just the continent.",
        'language':      "What is the main official language of the city '{entity}'? Answer with just the language.",
    },
    'song': {
        'performer':     "Who is the performer of the song '{entity}'? Answer with just the artist name.",
        'genre':         "What is the genre of the song '{entity}'? Answer with just the genre.",
        'release_year':  "In what year was the song '{entity}' released? Answer with just a year.",
    },
}

REFUSAL_PATTERNS = [
    r"i (?:don'?t|do not) (?:know|have)",
    r"i'?m (?:sorry|not sure|not familiar|unable)",
    r"i (?:cannot|can'?t) (?:provide|verify|confirm|find)",
    r"there (?:is|seems to be) (?:no|insufficient|limited) (?:information|data|record)",
    r"unable to (?:find|locate|verify)",
    r"(?:no|not enough|insufficient) (?:public(?:ly available)?\s+)?(?:information|data|record)",
    r"i don't have (?:specific|enough|reliable) (?:information|details|data)",
    r"there'?s no (?:widely|publicly|reliable) (?:known|available)",
]
REFUSAL_RE = re.compile('|'.join(REFUSAL_PATTERNS), re.IGNORECASE)

def is_refusal(text: str) -> bool:
    return bool(REFUSAL_RE.search(text or ''))

def normalise(s: str) -> str:
    return re.sub(r'[^a-z0-9]+', ' ', (s or '').lower()).strip()

def attr_match(answer: str, ground_truth) -> bool:
    if isinstance(ground_truth, list):
        return any(attr_match(answer, gt) for gt in ground_truth)
    if not ground_truth or not answer:
        return False
    a = normalise(str(answer)); g = normalise(str(ground_truth))
    if not g:
        return False
    # Substring match in either direction (handles "New York" matching "New York City")
    return g in a or a in g

def chat_query(question: str) -> str:
    messages = [{'role': 'user', 'content': question}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt')['input_ids'].to(device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=MAX_GEN_TOKENS, do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    answer = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
    return answer

labelled = []   # list of {'type', 'entity', 'class': 'known'|'unknown', 'n_correct', 'n_refuse'}
for ent in tqdm(candidates, desc='labelling'):
    name = ent['entity']
    type_ = ent['type']
    attrs = ent.get('attributes', [])
    # Map ground-truth by attribute_type (handle list values)
    gt_by_type = {}
    for a in attrs:
        gt_by_type.setdefault(a['attribute_type'], []).append(a['attribute_value'])

    templates = ATTRIBUTE_TEMPLATES.get(type_, {})
    available = [k for k in templates if k in gt_by_type]
    if len(available) < 2:
        continue

    selected = available[:N_ATTRIBUTES_TO_TEST]
    n_correct = 0; n_refuse = 0; answers = {}
    for attr_name in selected:
        q = templates[attr_name].format(entity=name)
        ans = chat_query(q)
        answers[attr_name] = ans
        if is_refusal(ans):
            n_refuse += 1
        elif attr_match(ans, gt_by_type[attr_name]):
            n_correct += 1

    n_total = len(selected)
    if n_correct >= 2 and n_refuse == 0:
        cls = 'known'
    elif n_correct == 0 and n_refuse >= 1:
        cls = 'unknown'
    else:
        cls = 'middle'

    labelled.append({
        'type': type_, 'entity': name, 'class': cls,
        'n_correct': n_correct, 'n_refuse': n_refuse, 'n_total': n_total,
        'sample_answers': answers,
    })

from collections import Counter
cnt = Counter(l['class'] for l in labelled)
by_type = Counter((l['type'], l['class']) for l in labelled)
print(f'\nlabelling done · total {len(labelled)} candidates')
for c, n in cnt.most_common():
    print(f'  {c}: {n}')
print('per-type:')
for (t, c), n in sorted(by_type.items()):
    print(f'  {t}/{c}: {n}')

## 4. Capture activations at last entity token

Same prompt-template structure as Ferrando — the entity is wrapped in single quotes inside a natural sentence. We capture the residual at the last token of the entity span.

In [ ]:
PROMPT_TEMPLATE = "What can you tell me about '{entity}'?"

_captured = {}
def capture_hook(key):
    def h(mod, inp, out):
        _captured[key] = out[0] if isinstance(out, tuple) else out
        return out
    return h

def find_entity_last_pos(prompt: str, entity: str) -> int:
    """Find the last token position belonging to the entity string inside the prompt.
    The entity is wrapped in single quotes; we locate the closing quote and step back."""
    full_ids = tok(prompt, return_tensors='pt')['input_ids'][0].tolist()
    # Find the closing single-quote sequence after the opening one
    open_q  = tok.encode("'", add_special_tokens=False)
    close_q = tok.encode("'?", add_special_tokens=False)
    n = len(full_ids)
    # Search for the closing pattern from the right
    m = len(close_q)
    for i in range(n - m, -1, -1):
        if full_ids[i:i+m] == close_q:
            return i - 1   # token right before the closing quote
    return n - 2

known_entries   = [l for l in labelled if l['class'] == 'known']
unknown_entries = [l for l in labelled if l['class'] == 'unknown']

def capture_z(entries, label):
    z_per_layer = {layer: [] for layer in LAYERS}
    hs = []
    for layer in LAYERS:
        hs.append(layer_mods[layer].register_forward_hook(capture_hook(layer)))
    with torch.no_grad():
        for entry in tqdm(entries, desc=label):
            prompt = PROMPT_TEMPLATE.format(entity=entry['entity'])
            ids = tok(prompt, return_tensors='pt')['input_ids'].to(device)
            pos = find_entity_last_pos(prompt, entry['entity'])
            _ = model(ids)
            for layer in LAYERS:
                resid = _captured[layer][0, pos].to(torch.bfloat16)
                z = saes[layer].encode(resid.unsqueeze(0))[0].float().cpu().numpy()
                z_per_layer[layer].append(z)
    for h in hs:
        h.remove()
    return {layer: np.stack(z_per_layer[layer], axis=0) for layer in LAYERS}

Z_known   = capture_z(known_entries,   'known   activations')
Z_unknown = capture_z(unknown_entries, 'unknown activations')

for layer in LAYERS:
    print(f'  L{layer}: Z_known={Z_known[layer].shape}, Z_unknown={Z_unknown[layer].shape}')

## 5. Pile noise filter — drop generic surface features

**This is the step v0.0.1 missed.** For each layer, run ~2k random Pile tokens through the SAE and compute per-feature activation rate. Any feature firing on >2% of Pile tokens is considered "generic" (likely a syntactic / token-level feature, not a knowledge feature) and is excluded from the entity-recognition probe.

Per Ferrando 2024 §3.1 + their `utils/activation_cache.py`.

In [ ]:
from datasets import load_dataset

pile_ds = load_dataset('NeelNanda/pile-10k', split='train', streaming=True)
pile_text = []
for ex in pile_ds:
    pile_text.append(ex['text'])
    if len(pile_text) >= 50:
        break

pile_tokens = tok(' '.join(pile_text), return_tensors='pt', max_length=N_PILE_TOKENS,
                  truncation=True)['input_ids'].to(device)
print(f'pile tokens: {pile_tokens.shape}')

fire_rate = {}
hs = []
for layer in LAYERS:
    hs.append(layer_mods[layer].register_forward_hook(capture_hook(layer)))
with torch.no_grad():
    _ = model(pile_tokens)
for h in hs:
    h.remove()

for layer in LAYERS:
    resid = _captured[layer][0]   # (T, D_MODEL)
    z = saes[layer].encode(resid.to(torch.bfloat16))   # (T, D_SAE)
    fire_rate[layer] = ((z > 0).float().mean(dim=0)).cpu().numpy()
    n_drop = int((fire_rate[layer] > PILE_FILTER_THRESHOLD).sum())
    print(f'  L{layer}: {n_drop:>5d}/{D_SAE} features active on >{PILE_FILTER_THRESHOLD*100:.1f}% of Pile (will be dropped)')

## 6. Per-feature separation + single-latent AUROC (after Pile filter)

Following Ferrando exactly: train/test split entities → compute Cohen's d on train only → drop Pile-noisy features → for each surviving top-100 feature compute single-latent AUROC on test set.

In [ ]:
from sklearn.metrics import roc_auc_score

n_k = len(known_entries); n_u = len(unknown_entries)
if n_k < 50 or n_u < 50:
    print(f'⚠ small dataset: known={n_k}, unknown={n_u}. AUROC CI will be wide.')

y = np.concatenate([np.ones(n_k), np.zeros(n_u)])
np.random.seed(SEED)
idx = np.arange(len(y)); np.random.shuffle(idx)
n_train = int(TRAIN_FRAC * len(y))
tr, te = idx[:n_train], idx[n_train:]

best_overall = {'layer': None, 'feature': None, 'auroc': 0.0}
auroc_per_layer = {}
top_features_per_layer = {}

for layer in LAYERS:
    X = np.concatenate([Z_known[layer], Z_unknown[layer]], axis=0)   # (N, D_SAE)
    Xk_tr = X[tr][y[tr] == 1]; Xu_tr = X[tr][y[tr] == 0]
    if len(Xk_tr) == 0 or len(Xu_tr) == 0:
        print(f'  L{layer}: empty class in train, skipping')
        continue
    mu_k, mu_u = Xk_tr.mean(0), Xu_tr.mean(0)
    sd = np.sqrt((Xk_tr.std(0)**2 + Xu_tr.std(0)**2) / 2) + 1e-9
    sep = (mu_k - mu_u) / sd
    # Pile filter: drop noisy features
    pile_mask = fire_rate[layer] <= PILE_FILTER_THRESHOLD
    sep_filtered = np.where(pile_mask, sep, 0.0)
    top_feats = np.argsort(-np.abs(sep_filtered))[:TOP_FEATURES_TO_TEST]
    top_features_per_layer[layer] = top_feats

    # Single-latent AUROC for each candidate feature on the test split
    aurocs = []
    for feat in top_feats:
        score = X[te, feat]   # raw activation as classifier score
        # Direction may be flipped (feature might fire MORE on unknown); take max(auroc, 1-auroc)
        auc = roc_auc_score(y[te], score)
        if auc < 0.5:
            auc = 1 - auc
        aurocs.append(auc)
    aurocs = np.array(aurocs)
    best_idx = int(np.argmax(aurocs))
    best_feat = int(top_feats[best_idx])
    best_auc = float(aurocs[best_idx])
    auroc_per_layer[layer] = {
        'best_feature':   best_feat,
        'best_auroc':     best_auc,
        'top10_mean':     float(np.sort(aurocs)[-10:].mean()),
        'sep_at_best':    float(sep_filtered[best_feat]),
        'fire_rate_at_best': float(fire_rate[layer][best_feat]),
    }
    print(f'  L{layer}: best single-latent AUROC = {best_auc:.4f}  (feature f{best_feat}, '
          f'sep={sep_filtered[best_feat]:+.3f}, pile_rate={fire_rate[layer][best_feat]*100:.2f}%)')
    if best_auc > best_overall['auroc']:
        best_overall = {'layer': layer, 'feature': best_feat, 'auroc': best_auc}

print(f'\nBEST overall: L{best_overall["layer"]} f{best_overall["feature"]}  AUROC={best_overall["auroc"]:.4f}')

## 7. Stage-gate decision + writeup

In [ ]:
best_auroc = best_overall['auroc']
if best_auroc >= 0.65:
    verdict, color = 'STRONG · entity-recognition signal real on Qwen3.6-27B', '\u2705'
elif best_auroc >= 0.55:
    verdict, color = 'WEAK · partial replication, document with caveats', '\u26a0\ufe0f'
else:
    verdict, color = 'NEGATIVE · no entity-recognition signal after Pile filter', '\u274c'

print(f'\n{"="*64}')
print(f'{color}  STAGE-GATE: {verdict}')
print(f'{"="*64}')
print(f'  best layer: L{best_overall["layer"]}  feature: f{best_overall["feature"]}  AUROC: {best_auroc:.4f}')
print(f'  reference (Ferrando 2024 on Gemma-2-2B-IT, L13): 0.732')
print(f'\n  per-layer best single-latent:')
for layer in LAYERS:
    if layer in auroc_per_layer:
        a = auroc_per_layer[layer]
        print(f'    L{layer}: AUROC={a["best_auroc"]:.4f}  feature=f{a["best_feature"]}  '
              f'top-10 mean={a["top10_mean"]:.4f}')

print(f'\n  dataset: {n_k} known + {n_u} unknown (after attribute-recall labelling)')
print(f'  test split: {(y[te] == 1).sum()} known, {(y[te] == 0).sum()} unknown')
print(f'  Pile filter dropped {int((fire_rate[best_overall["layer"]] > PILE_FILTER_THRESHOLD).sum())} features at best layer')

## 8. Inspect the best feature qualitatively

In [ ]:
import matplotlib.pyplot as plt

L = best_overall['layer']
f = best_overall['feature']
zk = Z_known[L][:, f]
zu = Z_unknown[L][:, f]

print(f'L{L}/f{f}  AUROC={best_auroc:.4f}')
print(f'  mean known   = {zk.mean():.3f}  std={zk.std():.3f}')
print(f'  mean unknown = {zu.mean():.3f}  std={zu.std():.3f}')
print(f'  pile fire rate = {fire_rate[L][f]*100:.2f}%')

print(f'\nTop 5 KNOWN entities by this feature\'s activation:')
for i in np.argsort(-zk)[:5]:
    e = known_entries[i]
    print(f'  {zk[i]:.3f}  [{e["type"]}]  {e["entity"]}')
print(f'\nTop 5 UNKNOWN entities by this feature\'s activation:')
for i in np.argsort(-zu)[:5]:
    e = unknown_entries[i]
    print(f'  {zu[i]:.3f}  [{e["type"]}]  {e["entity"]}')

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=140)
ax.hist(zk, bins=30, alpha=0.55, color='#16a34a', label=f'known (n={len(zk)})')
ax.hist(zu, bins=30, alpha=0.55, color='#f97316', label=f'unknown (n={len(zu)})')
ax.axvline(zk.mean(), color='#16a34a', linestyle='--', linewidth=1)
ax.axvline(zu.mean(), color='#f97316', linestyle='--', linewidth=1)
ax.set_xlabel('activation')
ax.set_ylabel('count')
ax.set_title(f'L{L}/f{f}  ·  single-latent AUROC={best_auroc:.4f}  (Pile-filtered)',
             fontsize=12, fontweight='bold')
ax.legend(frameon=False)
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.tight_layout()
chart_path = f'/tmp/halluc_v002_top.png'
plt.savefig(chart_path, dpi=140, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\nsaved {chart_path}')

## 9. Save artifacts to HF SAE repo

In [ ]:
from huggingface_hub import HfApi
from datetime import datetime, timezone

results = {
    'version':         'v0.0.2',
    'model':           HF_BASE_MODEL,
    'sae_repo':        HF_SAE_REPO,
    'method':          'Ferrando 2024 entity-recognition replication, properly controlled',
    'arxiv':           'arxiv:2411.14257',
    'ferrando_repo':   'https://github.com/javiferran/sae_entities',
    'dataset_source':  'real Wikidata entities from javiferran/sae_entities (no synthetic names)',
    'labelling':       'attribute-recall by Qwen3.6-27B (3 questions per entity, ≥2/3 correct = known, 0/3 + ≥1 refusal = unknown)',
    'capture_position': 'last entity token inside `What can you tell me about \'{entity}\'?`',
    'pile_filter':     {'n_pile_tokens': N_PILE_TOKENS, 'threshold': PILE_FILTER_THRESHOLD},
    'n_known':         n_k,
    'n_unknown':       n_u,
    'train_frac':      TRAIN_FRAC,
    'top_features_tested': TOP_FEATURES_TO_TEST,
    'auroc_per_layer': {
        str(layer): auroc_per_layer.get(layer, {})
        for layer in LAYERS
    },
    'best_layer':   best_overall['layer'],
    'best_feature': best_overall['feature'],
    'best_auroc':   best_auroc,
    'verdict':      verdict,
    'reference': {
        'ferrando_2024_gemma2_2b_it_l13_auroc': 0.732,
        'gate_threshold_strong': 0.65,
        'gate_threshold_weak':   0.55,
    },
    'v001_caveat':  ('v0.0.1 hit a tokenization confound (synthetic Slavic names ~2x token length of '
                     'famous names → AUROC=1.0 was surface signal, not entity recognition). v0.0.2 uses '
                     'real Wikidata entities only + Pile noise filter to fix that confound.'),
    'timestamp':    datetime.now(timezone.utc).isoformat(),
}

out_path = '/tmp/hallucination_v0_0_2.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

api = HfApi()
api.upload_file(
    path_or_fileobj=out_path,
    path_in_repo='hallucination_v0_0_2.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'Hallucination v0.0.2 — {verdict.split(chr(183))[0].strip()} (best AUROC={best_auroc:.3f} @ L{best_overall["layer"]})',
)
api.upload_file(
    path_or_fileobj=chart_path,
    path_in_repo='charts/hallucination_v002_top_feature.png',
    repo_id=HF_SAE_REPO,
    commit_message='Hallucination v0.0.2 chart — best entity-recognition feature (Pile-filtered)',
)
print(f'\n✓ uploaded → https://huggingface.co/{HF_SAE_REPO}')
print(f'\nVerdict: {verdict}')